# 2 — Preprocessing: yellow taxi trip records

**Input:** `data/landing/yellow_tripdata_YYYY-MM.parquet` (18 files, Jan 2023 – Jun 2024, 58,642,319 rows)

**Outputs:**

| File | Contents |
|---|---|
| `data/raw/trips_clean.parquet` | Full citywide cleaned trip records, partitioned by month. Consumed by notebook 3 (distribution and outlier analysis). |
| `data/curated/taxi_airport_hourly.parquet` | Table A — one row per `(pickup_date, pickup_hour, airport)` for JFK and LGA. 26,256 rows. |
| `data/curated/preprocessing_counts_taxi.csv` | Record count after each filter, for the preprocessing table in the report. |

## Order of operations, and why

Cleaning is applied to **all 58.6M rows before** the airport subset is taken, not after.
The subject specification requires the full distribution to be used when analysing
distributions, aggregating attributes, and performing outlier analysis. Filtering to two
taxi zones first would be cheaper, but the resulting outlier analysis would describe only
airport trips. Airport trips have an atypical fare distribution — the JFK flat fare
compresses a large mass of records onto a single value — so characterising them against
the citywide baseline is the more informative comparison, and it is the one the
specification asks for.

## Removal policy: two tiers

A record is **removed** only if it violates a documented business rule or is logically
impossible. A record that is merely extreme is **retained** and characterised in the
outlier analysis of notebook 3. Discarding extreme-but-possible records here would
pre-empt the outlier analysis and quietly narrow the distribution being reported.

In [1]:
"""Preprocessing of the TLC yellow taxi trip records.

Reads the raw monthly parquet files, harmonises their schemas, applies
business-rule filters to the full citywide dataset, and aggregates the airport
subset to hourly counts.
"""

import json
import sys
from functools import reduce
from operator import and_
from pathlib import Path

from pyspark.sql import DataFrame
from pyspark.sql import functions as F

# Session configuration, schema normalisation, and the verified zone
# identifiers live in `scripts/spark_utils.py` so that every notebook loads the
# data identically and the logic can be linted as ordinary source.
sys.path.append(str(Path("..") / "scripts"))
from spark_utils import (  # noqa: E402
    JFK_ZONE,
    LGA_ZONE,
    create_spark_session,
    load_trips,
)

# --- Paths -----------------------------------------------------------------
# Notebook is expected to run from `notebooks/`.
PROJECT_ROOT = Path("..").resolve()
LANDING_DIR = PROJECT_ROOT / "data" / "landing" / "tlc"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CURATED_DIR = PROJECT_ROOT / "data" / "curated"

for directory in (RAW_DIR, CURATED_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# --- Study window ----------------------------------------------------------
# Inclusive of the start date, exclusive of the end date.
WINDOW_START = "2023-01-01"
WINDOW_END = "2024-07-01"

# --- Taxi zones ------------------------------------------------------------
AIRPORT_ZONES = {JFK_ZONE: "JFK", LGA_ZONE: "LGA"}
# EWR (zone 1) is deliberately excluded. Yellow taxis may drop off at Newark but
# may not pick up there, so EWR enters this study only through the flight-side
# features built in notebook 2b.

In [2]:
spark = create_spark_session(app_name="MAST30034 — taxi preprocessing")
spark.sparkContext.setLogLevel("WARN")
spark.version

your 131072x1 screen size is bogus. expect trouble
26/08/16 18:23:46 WARN Utils: Your hostname, iphone resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/16 18:23:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/16 18:23:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'3.5.1'

## Step 1 — Load and harmonise the monthly files

`load_trips` reads every `yellow_tripdata_*.parquet` file in the landing directory,
passes each through `normalise_schema`, and unions them with `unionByName` so that a
difference in column ordering cannot silently misalign the frames.

`normalise_schema` handles the two schema differences across the eighteen months: the
airport surcharge is named `airport_fee` in some files and `Airport_fee` in others, and
the identifier and count columns vary between 64-bit and 32-bit integers and doubles.
Both are documented in the README.

In [3]:
trips_raw = load_trips(spark, LANDING_DIR)
print(f"{len(trips_raw.columns)} columns after normalisation")

19 columns after normalisation


In [4]:
# Derived quantities used by the filters below. Both are computed before any
# filtering so that they are available to the count waterfall in Step 3.
trips_raw = (
    trips_raw
    .withColumn(
        "trip_duration_s",
        F.unix_timestamp("tpep_dropoff_datetime")
        - F.unix_timestamp("tpep_pickup_datetime"),
    )
    .withColumn(
        "implied_mph",
        F.when(
            F.col("trip_duration_s") > 0,
            F.col("trip_distance") / (F.col("trip_duration_s") / 3600.0),
        ),
    )
)

## Step 2 — Filter definitions

Each filter below is stated with the rule it enforces. Thresholds that depend on the TLC
fare schedule are held in named constants so that the value used is visible in one place
and can be checked against the published schedule.

| # | Filter | Rule being enforced |
|---|---|---|
| 1 | Pickup inside the study window | Each monthly file contains a small number of records timestamped outside its own month, arising from meter and upload errors. Retaining them would misstate the timeline. |
| 2 | Drop-off after pickup | A trip cannot end before it starts. |
| 3 | Duration between 1 minute and 6 hours | Sub-minute records are meter mis-taps rather than journeys; multi-hour records are meters left running after the passenger has left. |
| 4 | Positive trip distance | A recorded distance of zero means no journey took place. |
| 5 | Positive fare and total | Negative amounts are voided transactions and chargebacks, not trips. |
| 6 | Implied speed under 80 mph | Not physically attainable across the five boroughs; indicates a corrupt odometer or timestamp reading. |
| 7 | Documented rate code | The data dictionary defines codes 1–6. Code 99 is undocumented and is removed. A rate code that is *missing* is retained and flagged — see below. |
| 8 | Documented payment type | As above: undocumented values removed, missing values retained and flagged. |

Two rules are recorded but **not** enforced as removals, because they do not invalidate
a record:

- `tip_amount` is populated for card payments only; cash tips are not recorded. Any tip
  statistic must therefore be computed over `payment_type = 1` alone. This is handled in
  the aggregation rather than by deleting cash trips, which are valid trips.
- Extreme-but-possible values — a 90-mile Rate Code 4 run to Westchester, a $200 fare —
  are retained for the outlier analysis.
- `ratecodeid` and `payment_type` are **missing** on a substantial minority of records,
  concentrated in the vendors that do not populate them. A missing value is not an
  undocumented one: the trip happened, and nothing about the record is invalid. Testing
  membership with `.isin()` would delete them silently, because the predicate evaluates
  to null and `.where()` drops null predicates — so the two filters above are written to
  admit nulls explicitly, and the affected records are flagged instead.

  This matters more than the row count suggests. `n_pickups` is the response variable,
  and these records are not distributed evenly across hours, airports, or vendors.
  Deleting them would push a vendor's reporting behaviour into the target and let it be
  read as demand. The cost of keeping them is that any statistic *derived from* the rate
  code must be computed over the records where it is known, which is why `share_flat_fare`
  carries its denominator as a column in Step 6.

In [5]:
# Minimum plausible metered fare. The initial charge under the fare schedule in
# force across the study window, before any surcharge. VERIFY against the TLC
# passenger-fare page and cite that page in the report before relying on this.
MIN_METERED_FARE = 3.00

MAX_IMPLIED_MPH = 80.0
MIN_DURATION_S = 60
MAX_DURATION_S = 6 * 3600

# The data dictionary defines rate codes 1-6 and payment types 1-6. Code 99
# appears in the feed and is documented nowhere; it is removed. Missing values
# are a separate case and are retained — see the note in Step 2.
VALID_RATECODES = [1, 2, 3, 4, 5, 6]
VALID_PAYMENT_TYPES = [1, 2, 3, 4, 5, 6]

# Ordered list of (label, predicate). The label is reproduced verbatim in the
# preprocessing table of the report, so keep the two in step.
FILTERS = [
    (
        "Pickup within study window",
        (F.col("tpep_pickup_datetime") >= F.lit(WINDOW_START).cast("timestamp"))
        & (F.col("tpep_pickup_datetime") < F.lit(WINDOW_END).cast("timestamp")),
    ),
    (
        "Drop-off after pickup",
        F.col("trip_duration_s") > 0,
    ),
    (
        "Duration between 1 min and 6 h",
        (F.col("trip_duration_s") >= MIN_DURATION_S)
        & (F.col("trip_duration_s") <= MAX_DURATION_S),
    ),
    (
        "Positive trip distance",
        F.col("trip_distance") > 0,
    ),
    (
        "Positive fare and total",
        (F.col("fare_amount") > 0) & (F.col("total_amount") > 0),
    ),
    (
        "Metered fare at or above initial charge",
        # `eqNullSafe` rather than `==` so that a missing rate code yields
        # False rather than null here: `null != 1` is null, which `.where()`
        # would drop, deleting the very records this filter does not judge.
        ~F.col("ratecodeid").eqNullSafe(1) | (F.col("fare_amount") >= MIN_METERED_FARE),
    ),
    (
        "Implied speed below 80 mph",
        F.col("implied_mph") < MAX_IMPLIED_MPH,
    ),
    (
        "Documented rate code",
        F.col("ratecodeid").isNull() | F.col("ratecodeid").isin(VALID_RATECODES),
    ),
    (
        "Documented payment type",
        F.col("payment_type").isNull()
        | F.col("payment_type").isin(VALID_PAYMENT_TYPES),
    ),
]

## Step 3 — Record counts after each filter

The specification requires the dataset shape to be stated at each filtering step, and the
figures are cross-checked against this code during marking.

Calling `.count()` after each of the nine filters would trigger nine full scans of 58.6M
rows. Instead, the cumulative conjunction of the predicates is evaluated in a **single
pass**: summing `predicate_1`, then `predicate_1 AND predicate_2`, and so on, gives
exactly the sequential post-filter counts for one scan of the data.

`F.when(condition, 1).otherwise(0)` is used rather than casting the boolean, so that a
null predicate — arising from a null `passenger_count`, for instance — counts as an
exclusion. This matches the behaviour of `.where()`, which also drops null predicates.

In [6]:
def count_waterfall(df: DataFrame, filters: list) -> list:
    """Compute the row count remaining after each filter, in a single pass.

    Args:
        df: The unfiltered DataFrame.
        filters: Ordered list of `(label, predicate)` pairs.

    Returns:
        List of `(label, rows_remaining, rows_removed)` tuples, preceded by an
        entry for the raw ingest.
    """
    cumulative = F.lit(True)
    aggregations = [F.count("*").alias("stage_0")]
    for i, (_, predicate) in enumerate(filters, start=1):
        cumulative = cumulative & predicate
        aggregations.append(
            F.sum(F.when(cumulative, 1).otherwise(0)).alias(f"stage_{i}")
        )

    row = df.agg(*aggregations).collect()[0]

    waterfall = [("Raw ingest", row["stage_0"], None)]
    for i, (label, _) in enumerate(filters, start=1):
        remaining = row[f"stage_{i}"]
        waterfall.append((label, remaining, row[f"stage_{i - 1}"] - remaining))
    return waterfall


waterfall = count_waterfall(trips_raw, FILTERS)

for label, remaining, removed in waterfall:
    removed_str = "—" if removed is None else f"{removed:,}"
    print(f"{label:<42} {remaining:>12,}  removed {removed_str:>10}")

Raw ingest                                   58,642,319  removed          —
Pickup within study window                   58,642,192  removed        127
Drop-off after pickup                        58,620,446  removed     21,746
Duration between 1 min and 6 h               57,922,347  removed    698,099
Positive trip distance                       57,243,639  removed    678,708
Positive fare and total                      56,642,200  removed    601,439
Metered fare at or above initial charge      56,641,660  removed        540
Implied speed below 80 mph                   56,637,949  removed      3,711
Documented rate code                         56,244,911  removed    393,038
Documented payment type                      53,451,022  removed  2,793,889


In [7]:
# Persist for the report table. Retention is reported against the raw ingest.
raw_total = waterfall[0][1]
rows = [
    {
        "step": label,
        "rows": remaining,
        "removed": removed,
        "pct_of_raw": round(100 * remaining / raw_total, 3),
    }
    for label, remaining, removed in waterfall
]

counts_path = CURATED_DIR / "preprocessing_counts_taxi.csv"
spark.createDataFrame(rows).coalesce(1).toPandas().to_csv(counts_path, index=False)
print(f"Written to {counts_path}")

Written to /home/tavish/projects/project-1-individual-SavvyHack/data/curated/preprocessing_counts_taxi.csv


## Step 4 — Apply the filters and persist the cleaned citywide dataset

This is the dataset that notebook 3 uses for the distribution and outlier analysis. It is
written before the airport subset is taken, which is the point of the ordering decision
made at the top of this notebook.

It is partitioned by month so that the analysis notebook can read a subset without a full
scan, and so that any month-specific data quality problem is isolated to one partition.

In [8]:
trips_clean = (
    trips_raw
    .where(reduce(and_, (predicate for _, predicate in FILTERS)))
    .withColumn("pickup_month", F.date_format("tpep_pickup_datetime", "yyyy-MM"))
    # Retained, but marked, so that any statistic derived from these two
    # columns can state its own denominator rather than assume one.
    .withColumn("ratecode_missing", F.col("ratecodeid").isNull())
    .withColumn("payment_type_missing", F.col("payment_type").isNull())
)

(
    trips_clean
    .write
    .mode("overwrite")
    .partitionBy("pickup_month")
    .parquet(str(RAW_DIR / "trips_clean.parquet"))
)
print("Cleaned citywide dataset written.")

Cleaned citywide dataset written.


In [9]:
# Read back from disk rather than recomputing the filter chain for every
# downstream action. Without this, each subsequent stage re-reads and re-filters
# all 18 monthly files.
trips_clean = spark.read.parquet(str(RAW_DIR / "trips_clean.parquet"))
clean_total = trips_clean.count()
assert clean_total == waterfall[-1][1], (
    f"Written {clean_total:,} rows but waterfall predicted {waterfall[-1][1]:,}"
)
print(f"{clean_total:,} rows verified on disk")

53,451,022 rows verified on disk


## Step 5 — Airport subset

Pickups at JFK (zone 132) and LaGuardia (zone 138) are extracted. Only the **pickup**
zone is used: the research question concerns whether a driver should join an airport
queue, so a trip *to* the airport is not part of the target.

### Independent validation of the zone filter

Rate Code 2 denotes the JFK flat fare, and is recorded by the meter independently of the
GPS-derived `PULocationID`. Almost every Rate Code 2 trip should therefore touch zone 132
at one end. Agreement between the two fields is evidence that the zone filter is
selecting what it is meant to select; a large disagreement would indicate that the zone
identifiers are wrong.

In [10]:
# Cross-check: what share of Rate Code 2 trips touch zone 132 at either end?
ratecode_2 = trips_clean.where(F.col("ratecodeid") == 2)
validation = ratecode_2.agg(
    F.count("*").alias("ratecode_2_trips"),
    F.avg(
        F.when(
            (F.col("pulocationid") == JFK_ZONE)
            | (F.col("dolocationid") == JFK_ZONE),
            1.0,
        ).otherwise(0.0)
    ).alias("share_touching_zone_132"),
).collect()[0]

print(f"Rate Code 2 trips:            {validation['ratecode_2_trips']:,}")
print(f"Share touching zone 132:      {validation['share_touching_zone_132']:.4f}")

Rate Code 2 trips:            2,019,422
Share touching zone 132:      0.9619


In [11]:
airport_trips = (
    trips_clean
    .where(F.col("pulocationid").isin(list(AIRPORT_ZONES)))
    .withColumn(
        "airport",
        F.when(F.col("pulocationid") == JFK_ZONE, F.lit("JFK"))
        .otherwise(F.lit("LGA")),
    )
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
)

airport_total = airport_trips.count()
print(f"Airport pickups: {airport_total:,} "
      f"({100 * airport_total / clean_total:.2f}% of cleaned trips)")
airport_trips.groupBy("airport").count().show()

Airport pickups: 4,637,237 (8.68% of cleaned trips)


+-------+-------+
|airport|  count|
+-------+-------+
|    JFK|2746141|
|    LGA|1891096|
+-------+-------+



## Step 6 — Aggregate to Table A

One row per `(pickup_date, pickup_hour, airport)`.

`n_pickups` is the modelling target. The remaining columns describe the trips that
occurred in that hour and are used for the descriptive analysis and to construct
**lagged** features later. They must not be used as same-hour model inputs: the number of
card payments in hour *h* is not knowable to a driver deciding whether to join the queue
for hour *h*.

`mean_tip_ratio` is computed over card payments only, because cash tips are not recorded.
`share_flat_fare` is computed over the trips whose rate code is known, and
`n_ratecode_known` carries that denominator so the report never has to infer it.

In [12]:
hourly = airport_trips.groupBy("pickup_date", "pickup_hour", "airport").agg(
    F.count("*").alias("n_pickups"),
    F.avg("fare_amount").alias("mean_fare"),
    F.expr("percentile_approx(fare_amount, 0.5)").alias("median_fare"),
    F.avg("total_amount").alias("mean_total"),
    F.avg("trip_distance").alias("mean_distance_mi"),
    F.avg(F.col("trip_duration_s") / 60.0).alias("mean_duration_min"),
    F.avg("passenger_count").alias("mean_passengers"),
    # Share of pickups on the JFK flat fare, a proxy for Manhattan-bound
    # demand, over the trips whose rate code is known. The third branch is
    # left unset so that a missing rate code contributes to neither the
    # numerator nor the denominator, rather than being counted as "not flat".
    F.count("ratecodeid").alias("n_ratecode_known"),
    F.avg(
        F.when(F.col("ratecodeid") == 2, 1.0)
        .when(F.col("ratecodeid").isNotNull(), 0.0)
    ).alias("share_flat_fare"),
    F.sum(F.when(F.col("payment_type") == 1, 1).otherwise(0)).alias("n_card_trips"),
    # Tips are recorded for card payments only (data dictionary), so the mean is
    # taken over card trips alone rather than over all trips.
    F.avg(
        F.when(F.col("payment_type") == 1, F.col("tip_amount") / F.col("fare_amount"))
    ).alias("mean_tip_ratio"),
)
print(f"{hourly.count():,} non-empty airport-hours")

25,009 non-empty airport-hours


### Hours with no pickups

An airport-hour with zero pickups produces no group, so it is absent from the
aggregation above. Dropping those rows would remove precisely the low-demand hours the
model needs to learn — a count model fitted only to hours that had trips will
systematically overpredict the overnight period.

A complete spine of every `(date, hour, airport)` combination is therefore constructed and
left-joined. `n_pickups` and `n_card_trips` are filled with zero. The mean columns are
left **null**: the mean fare of no trips is undefined, not zero, and filling it would
inject a spurious cluster at the origin into every fare plot.

In [13]:
spine = (
    spark.sql(
        f"SELECT explode(sequence(to_date('{WINDOW_START}'), "
        f"date_sub(to_date('{WINDOW_END}'), 1), interval 1 day)) AS pickup_date"
    )
    .crossJoin(spark.range(24).select(F.col("id").cast("int").alias("pickup_hour")))
    .crossJoin(
        spark.createDataFrame([(code,) for code in AIRPORT_ZONES.values()], ["airport"])
    )
)

expected_rows = 547 * 24 * 2  # 547 days in the window, 24 hours, 2 airports
assert spine.count() == expected_rows, "Spine does not cover the study window"

table_a = (
    spine
    .join(hourly, ["pickup_date", "pickup_hour", "airport"], how="left")
    .fillna(0, subset=["n_pickups", "n_card_trips", "n_ratecode_known"])
)

### Daylight saving

Timestamps are New York wall-clock time, so the spine contains two kinds of anomalous
hour:

- **Spring forward** (2023-03-12, 2024-03-10): the 02:00 hour does not exist. The spine
  creates a row for it and the left join fills it with zero pickups, which is an artefact
  rather than an observation of no demand.
- **Fall back** (2023-11-05): the 01:00 hour occurs twice, so that row contains two
  hours' worth of trips and will read as an outlier.

Six rows out of 26,256 are affected. They are flagged rather than deleted so that the
modelling notebook can exclude them explicitly and the report can state that it did.

In [14]:
DST_SPRING_FORWARD = ["2023-03-12", "2024-03-10"]  # 02:00 does not exist
DST_FALL_BACK = ["2023-11-05"]                      # 01:00 occurs twice

table_a = table_a.withColumn(
    "dst_anomaly",
    (
        F.col("pickup_date").cast("string").isin(DST_SPRING_FORWARD)
        & (F.col("pickup_hour") == 2)
    )
    | (
        F.col("pickup_date").cast("string").isin(DST_FALL_BACK)
        & (F.col("pickup_hour") == 1)
    ),
)

table_a.where(F.col("dst_anomaly")).select(
    "pickup_date", "pickup_hour", "airport", "n_pickups"
).orderBy("pickup_date", "airport").show()

+-----------+-----------+-------+---------+
|pickup_date|pickup_hour|airport|n_pickups|
+-----------+-----------+-------+---------+
| 2023-03-12|          2|    JFK|        0|
| 2023-03-12|          2|    LGA|        0|
| 2023-11-05|          1|    JFK|       47|
| 2023-11-05|          1|    LGA|        0|
| 2024-03-10|          2|    JFK|        0|
| 2024-03-10|          2|    LGA|        0|
+-----------+-----------+-------+---------+



## Step 7 — Validate and write

Three checks before Table A leaves this notebook. Each one catches a failure that would
otherwise surface as a plausible-looking but wrong number much later.

In [15]:
# 1. Every airport-hour in the window is present exactly once.
assert table_a.count() == expected_rows
assert table_a.dropDuplicates(
    ["pickup_date", "pickup_hour", "airport"]
).count() == expected_rows

# 2. No trip was lost or duplicated by the grouping and the join.
assert table_a.agg(F.sum("n_pickups")).collect()[0][0] == airport_total

# 3. The flat-fare share is defined in exactly the hours where a rate code was
#    observed, and nowhere else.
mismatched = table_a.where(
    F.col("share_flat_fare").isNull() != (F.col("n_ratecode_known") == 0)
).count()
assert mismatched == 0, f"{mismatched} rows disagree on the rate code denominator"

# 4. Report how many genuine zero-demand hours exist, excluding the DST artefacts.
zero_hours = table_a.where(
    (F.col("n_pickups") == 0) & (~F.col("dst_anomaly"))
).count()
print(f"Genuine zero-pickup airport-hours: {zero_hours} of {expected_rows}")
print("All checks passed.")

Genuine zero-pickup airport-hours: 1242 of 26256
All checks passed.


In [16]:
(
    table_a
    .orderBy("pickup_date", "pickup_hour", "airport")
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet(str(CURATED_DIR / "taxi_airport_hourly.parquet"))
)

table_a.orderBy("pickup_date", "pickup_hour", "airport").show(6, truncate=False)

+-----------+-----------+-------+---------+------------------+-----------+-----------------+------------------+------------------+------------------+----------------+-------------------+------------+-------------------+-----------+
|pickup_date|pickup_hour|airport|n_pickups|mean_fare         |median_fare|mean_total       |mean_distance_mi  |mean_duration_min |mean_passengers   |n_ratecode_known|share_flat_fare    |n_card_trips|mean_tip_ratio     |dst_anomaly|
+-----------+-----------+-------+---------+------------------+-----------+-----------------+------------------+------------------+------------------+----------------+-------------------+------------+-------------------+-----------+
|2023-01-01 |0          |JFK    |251      |56.86892430278884 |60.4       |70.02266932270913|14.393386454183268|26.930677290836673|1.5617529880478087|251             |0.26693227091633465|160         |0.1871930365224332 |false      |
|2023-01-01 |0          |LGA    |15       |37.580000000000005|38.0      

In [17]:
# Record the airport-side shapes alongside the citywide waterfall so that the
# preprocessing table in the report is complete.
airport_shapes = {
    "cleaned_citywide_trips": clean_total,
    "airport_pickups": airport_total,
    "non_empty_airport_hours": hourly.count(),
    "table_a_rows": expected_rows,
    "genuine_zero_pickup_hours": zero_hours,
    # Retained rather than deleted; reported so the report can say how many.
    "airport_pickups_missing_ratecode": (
        airport_total
        - table_a.agg(F.sum("n_ratecode_known")).collect()[0][0]
    ),
}
with open(CURATED_DIR / "shapes_taxi.json", "w") as handle:
    json.dump(airport_shapes, handle, indent=2)

airport_shapes

{'cleaned_citywide_trips': 53451022,
 'airport_pickups': 4637237,
 'non_empty_airport_hours': 25009,
 'table_a_rows': 26256,
 'genuine_zero_pickup_hours': 1242,
 'airport_pickups_missing_ratecode': 0}

In [18]:
spark.stop()

## What to carry into the report

- The count waterfall printed in Step 3 populates the preprocessing table. Quote the
  labels from `FILTERS` verbatim so the table and the code agree.
- The Rate Code 2 agreement figure from Step 5 justifies the airport zone identifiers
  without asking the reader to take them on trust.
- The share of cleaned trips that are airport pickups justifies the narrowing from
  citywide to airport, and the fact that the citywide dataset was cleaned first justifies
  the outlier analysis in notebook 3.
- The DST and zero-demand-hour handling are each worth one sentence in the preprocessing
  section. They are small, but they are the kind of decision a marker checks the code for.
- The records with a missing rate code are retained and flagged rather than deleted. State
  the count, and state that `share_flat_fare` is computed over the known subset — it is a
  one-line justification that pre-empts an obvious question about the denominator.

**Next:** notebook 2b — flight-side aggregation to the same `(date, hour, airport)` key,
then the join and the temporal and weather features.